# Export well-known brands (one bucket per gender x brand)

The opposite of `00_brand_filter.ipynb`: instead of *dropping* the well-known brands,
this exports **each (gender x well-known brand) into its own folder**, e.g.
`data/women_nike/`, `data/men_adidas/`.

Each folder gets `amazon_shoes_8_combined.parquet` (the bucket-filtered raw), the same
input format the `data_preparation/` pipeline expects. After running this, point
`data_preparation/00_part1...` (and the rest) at any of these new buckets by setting
`BRAND_BUCKET = 'women_nike'` etc.

Note: `men_nike` is the same data as the existing `men_top2_nike`; this notebook just
re-exports it under the gender_brand name for consistency. It does NOT touch the
existing `*_top1_*`, `*_top2_*`, `*_other` buckets.

In [11]:
import pandas as pd
import re
from pathlib import Path

def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data_preparation').exists() and (p / 'code').exists():
            return p
    raise RuntimeError(
        'Cannot find project root.\n'
        'Expected a directory containing both "data_preparation/" and "code/".\n'
        f'Searched from: {Path.cwd()}'
    )

PROJECT_ROOT = _find_project_root()
DATA_ROOT    = PROJECT_ROOT / 'data'
INPUT_FILE   = DATA_ROOT / 'amazon_shoes_8_combined.parquet'

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'INPUT_FILE   : {INPUT_FILE}')
assert INPUT_FILE.exists(), f'Missing input file: {INPUT_FILE}'

PROJECT_ROOT : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate
INPUT_FILE   : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/amazon_shoes_8_combined.parquet


In [12]:
df = pd.read_parquet(INPUT_FILE)
df['date'] = pd.to_datetime(df['date'])

asin_df = df.drop_duplicates('ASIN')[['ASIN', 'brand', 'gender']]

print(f'Total rows   : {len(df):,}')
print(f'Unique ASINs : {df["ASIN"].nunique():,}')
print(f'Unique brands: {df["brand"].nunique():,}')
print()
print('Gender breakdown (ASIN-level):')
for g, n in asin_df['gender'].value_counts().items():
    print(f'  {g:<40} {n:>6}')

Total rows   : 1,008,930
Unique ASINs : 7,761
Unique brands: 664

Gender breakdown (ASIN-level):
  Women                                      5472
  Men                                        2279
  Costumes & Accessories                        9
  Shoe, Jewelry & Watch Accessories             1


## Which brands to export

Same curated well-known list as `00_brand_filter.ipynb`, per gender so you can adjust
one gender without touching the other. Edit these lists to add/remove brands. Only
brands that actually appear in a gender's data get a folder.

In [13]:
_DEFAULT_WELL_KNOWN = [
    'Nike',
    'adidas',
    'ASICS',
    'Reebok',
    'Skechers',
    'PUMA',
    'New Balance',
    'Converse',
    'Vans',
    'HEYDUDE',
    'OLUKAI',
    "Dr.Scholl's SHOES",
    'Crocs',
    'UGG',
    'Timberland',
    'Clarks',
    'Under Armour',
    'Amazon Essentials',
]

WOMEN_WELL_KNOWN = list(_DEFAULT_WELL_KNOWN)
MEN_WELL_KNOWN   = list(_DEFAULT_WELL_KNOWN)

WELL_KNOWN_BY_GENDER = {
    'Women': WOMEN_WELL_KNOWN,
    'Men':   MEN_WELL_KNOWN,
}

GENDERS_TO_PROCESS = ['Women', 'Men']

# Only export brands with AT LEAST this many unique ASINs.
# (smaller brands are too noisy for elasticity and trigger missing-cluster errors)
MIN_ASINS = 40

for g, lst in WELL_KNOWN_BY_GENDER.items():
    print(f'{g:<6}: {len(lst)} well-known brands listed')

Women : 18 well-known brands listed
Men   : 18 well-known brands listed


In [14]:
print('=' * 70)
print('AUDIT: ASIN count of each well-known brand within each gender')
print('(0 = not in this gender; those are skipped)')
print('=' * 70)

for g in GENDERS_TO_PROCESS:
    g_counts = asin_df[asin_df['gender'] == g]['brand'].value_counts()
    print(f'\n--- {g} ---')
    print(f'  {"brand":<25} {"ASINs":>7}')
    for b in WELL_KNOWN_BY_GENDER[g]:
        n = int(g_counts.get(b, 0))
        flag = '  (not in data)' if n == 0 else ('  (skip: <= 40)' if n <= 40 else '')
        print(f'  {b:<25} {n:>7}{flag}')

AUDIT: ASIN count of each well-known brand within each gender
(0 = not in this gender; those are skipped)

--- Women ---
  brand                       ASINs
  Nike                           83
  adidas                         37  (skip: <= 40)
  ASICS                          47
  Reebok                         40  (skip: <= 40)
  Skechers                       23  (skip: <= 40)
  PUMA                           15  (skip: <= 40)
  New Balance                     0  (not in data)
  Converse                        0  (not in data)
  Vans                            0  (not in data)
  HEYDUDE                        44
  OLUKAI                         21  (skip: <= 40)
  Dr.Scholl's SHOES              65
  Crocs                           0  (not in data)
  UGG                             0  (not in data)
  Timberland                      0  (not in data)
  Clarks                          0  (not in data)
  Under Armour                    3  (skip: <= 40)
  Amazon Essentials              16 

In [15]:
def slugify(s: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', s.lower()).strip('_')

summary_rows = []

for g in GENDERS_TO_PROCESS:
    g_df     = df[df['gender'] == g]
    g_counts = g_df.drop_duplicates('ASIN')['brand'].value_counts()

    for brand in WELL_KNOWN_BY_GENDER[g]:
        n_asin = int(g_counts.get(brand, 0))
        if n_asin < MIN_ASINS:
            if n_asin == 0:
                print(f'skip {g}/{brand}: not in this gender')
            else:
                print(f'skip {g}/{brand}: {n_asin} ASINs (<= {MIN_ASINS}; need > {MIN_ASINS})')
            continue

        bucket_df  = g_df[g_df['brand'] == brand]
        folder_name = f'{g.lower()}_{slugify(brand)}'
        out_dir    = DATA_ROOT / folder_name
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path   = out_dir / 'amazon_shoes_8_combined.parquet'
        bucket_df.to_parquet(out_path, index=False)

        summary_rows.append({
            'folder' : out_path.parent.relative_to(PROJECT_ROOT).as_posix(),
            'gender' : g,
            'brand'  : brand,
            'rows'   : len(bucket_df),
            'asins'  : bucket_df['ASIN'].nunique(),
            'size_mb': round(out_path.stat().st_size / 1e6, 1),
        })

summary = pd.DataFrame(summary_rows)
print()
print(summary.to_string(index=False))
print()
print(f'Buckets written: {len(summary)}')
print(f'Rows written   : {summary["rows"].sum():,}')
print(f'ASINs written  : {summary["asins"].sum():,}')

skip Women/adidas: 37 ASINs (<= 40; need > 40)
skip Women/Skechers: 23 ASINs (<= 40; need > 40)
skip Women/PUMA: 15 ASINs (<= 40; need > 40)
skip Women/New Balance: not in this gender
skip Women/Converse: not in this gender
skip Women/Vans: not in this gender
skip Women/OLUKAI: 21 ASINs (<= 40; need > 40)
skip Women/Crocs: not in this gender
skip Women/UGG: not in this gender
skip Women/Timberland: not in this gender
skip Women/Clarks: not in this gender
skip Women/Under Armour: 3 ASINs (<= 40; need > 40)
skip Women/Amazon Essentials: 16 ASINs (<= 40; need > 40)
skip Men/Skechers: 26 ASINs (<= 40; need > 40)
skip Men/PUMA: 31 ASINs (<= 40; need > 40)
skip Men/New Balance: 7 ASINs (<= 40; need > 40)
skip Men/Converse: not in this gender
skip Men/Vans: not in this gender
skip Men/Dr.Scholl's SHOES: 7 ASINs (<= 40; need > 40)
skip Men/Crocs: not in this gender
skip Men/UGG: not in this gender
skip Men/Timberland: not in this gender
skip Men/Clarks: not in this gender
skip Men/Under Armour